In [ ]:
from google.colab import drive
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, GlobalAveragePooling1D,
    Dense, Dropout, Concatenate, ReLU, GaussianNoise, SpatialDropout1D
)
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Mount Drive
drive.mount('/content/drive')

# Create Output Directories
OUTPUT_DIR = '/content/drive/MyDrive/HeatSense/output'
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
# --- Data Settings ---
RAW_DATA_PATH = '/content/sen_ds_raw.xlsx' # Adjust path as needed
TARGET = 'Corerectal'
DYNAMIC_RAW_FEATURES = ['HR', 'SkinTemp_UpperArm', 'Env_Temp', 'Env_Humidity']
STATIC_FEATURES = ['Age', 'Sex_Encoded', 'Body_Mass_Kg', 'Height_Cm', 'Body_Fat_Pct', 'VO2Peak']
ENGINEERED_FEATURES = ['HR_slope_5m', 'SkinTemp_slope_5m']
DYNAMIC_FEATURES = DYNAMIC_RAW_FEATURES + ENGINEERED_FEATURES

# --- Model Hyperparameters ---
WINDOW_SIZE = 30      # 30 minutes
TRAIN_STRIDE = 1      # Overlapping for training
EVAL_STRIDE = 30      # Non-overlapping for honest test evaluation
DROPOUT_RATE = 0.35
L2_REG = 1e-4
BATCH_SIZE = 32
MAX_EPOCHS = 200
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

In [ ]:
def load_and_split_data(path):
    df = pd.read_excel(path, sheet_name='Data', header=[0, 1])

    # Flatten Headers
    new_cols = []
    for col in df.columns:
        lvl0, lvl1 = str(col[0]).strip(), str(col[1]).strip()
        new_cols.append(lvl1 if 'Unnamed' in lvl0 else (lvl0 if lvl0 == lvl1 else f"{lvl0}_{lvl1}"))
    df.columns = [c.split('_Unnamed')[0].strip().replace(' ', '_') for c in new_cols]
    df.replace([9999, 9999.0, '9999', '9999.0'], np.nan, inplace=True)

    # Identify Sessions
    df['Time_Sec'] = pd.to_timedelta(df['Time'].astype(str), errors='coerce').dt.total_seconds()
    df['Session_ID'] = ((df['Participant'] != df['Participant'].shift(1)) |
                        (df['Time_Sec'] < df['Time_Sec'].shift(1))).cumsum()

    # Participant-Wise Split
    unique_parts = df['Participant'].unique()
    train_p, temp_p = train_test_split(unique_parts, test_size=0.3, random_state=RANDOM_SEED)
    val_p, test_p = train_test_split(temp_p, test_size=0.5, random_state=RANDOM_SEED)

    return df, train_p, val_p, test_p

def process_splits(df, train_p, val_p, test_p):
    df_train = df[df['Participant'].isin(train_p)].copy()
    df_val = df[df['Participant'].isin(val_p)].copy()
    df_test = df[df['Participant'].isin(test_p)].copy()

    # 1. Impute Static (Fit on Train Only)
    imputer = KNNImputer(n_neighbors=5).fit(df_train[STATIC_FEATURES])
    df_train[STATIC_FEATURES] = imputer.transform(df_train[STATIC_FEATURES])
    df_val[STATIC_FEATURES] = imputer.transform(df_val[STATIC_FEATURES])
    df_test[STATIC_FEATURES] = imputer.transform(df_test[STATIC_FEATURES])

    # 2. Engineer Features (Slopes)
    def add_slopes(group):
        group['HR_slope_5m'] = group['HR'].diff(5).fillna(0)
        group['SkinTemp_slope_5m'] = group['SkinTemp_UpperArm'].diff(5).fillna(0)
        return group

    df_train = df_train.groupby('Session_ID').apply(add_slopes)
    df_val = df_val.groupby('Session_ID').apply(add_slopes)
    df_test = df_test.groupby('Session_ID').apply(add_slopes)

    # 3. Scale (Fit on Train Only)
    scaler = StandardScaler().fit(df_train[DYNAMIC_FEATURES + STATIC_FEATURES])
    df_train[DYNAMIC_FEATURES + STATIC_FEATURES] = scaler.transform(df_train[DYNAMIC_FEATURES + STATIC_FEATURES])
    df_val[DYNAMIC_FEATURES + STATIC_FEATURES] = scaler.transform(df_val[DYNAMIC_FEATURES + STATIC_FEATURES])
    df_test[DYNAMIC_FEATURES + STATIC_FEATURES] = scaler.transform(df_test[DYNAMIC_FEATURES + STATIC_FEATURES])

    return df_train, df_val, df_test

In [ ]:
def build_heatsense_model():
    # Dynamic Branch (CNN)
    input_dyn = Input(shape=(WINDOW_SIZE, len(DYNAMIC_FEATURES)), name='dynamic_input')
    x = GaussianNoise(0.05)(input_dyn)
    x = Conv1D(32, 5, padding='same', kernel_regularizer=l2(L2_REG))(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv1D(64, 3, padding='same', kernel_regularizer=l2(L2_REG))(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = GlobalAveragePooling1D()(x)

    # Static Branch
    input_stat = Input(shape=(len(STATIC_FEATURES),), name='static_input')
    s = Dense(16, kernel_regularizer=l2(L2_REG))(input_stat)
    s = BatchNormalization()(s)
    s = ReLU()(s)

    # Merge
    merged = Concatenate()([x, s])
    z = Dense(32, kernel_regularizer=l2(L2_REG))(merged)
    z = ReLU()(z)
    z = Dropout(DROPOUT_RATE)(z)
    output = Dense(1, activation='linear', name='tcore_pred')(z)

    model = Model(inputs=[input_dyn, input_stat], outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

Loading raw data...
  Excluded P1-P18: 17264 -> 11024 rows (22 participants remain)
  Dropped 56 rows with missing Corerectal
  Final: 10968 rows, 102 sessions, 22 participants
  Target (Corerectal): mean=37.85, std=0.54, range=[35.96, 39.45]
  Remaining NaN in dynamic features (to be imputed after split):
    HR: 351 (3.2%)
    SkinTemp_UpperArm: 29 (0.3%)
    Env_Temp: 227 (2.1%)
    Env_Humidity: 227 (2.1%)

Column dtypes:
Session_ID               int64
Participant              int64
Time_Sec               float64
HR                     float32
SkinTemp_UpperArm      float32
Env_Temp               float32
Env_Humidity           float32
Age                    float32
Sex                      int64
Body_Mass_Kg           float64
Height_Cm              float64
Body_Fat_Pct           float64
VO2Peak                float32
Acclimation_Status       int64
Training_Status          int64
Sex_Encoded            float32
BMI                    float32
Acclimation_Encoded    float32
Training_Enc

In [ ]:
"""
HeatSense Split & Scale Pipeline (Colab Version)
================================================
Implements the correct processing order: SPLIT -> IMPUTE -> ENGINEER -> SCALE -> WINDOW.

All statistics (medians, scaler params, KNN neighbors) are computed from the
training set only and applied to val/test, preventing any data leakage.

NOTE: Run the Configuration cell AND the Data Pipeline cell BEFORE running this cell.
"""

import numpy as np
import pandas as pd
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

# ---------------------------------------------------------------------------
# 1. SPLIT — by participant, stratified by whether they reach Tcore > 39.0
# ---------------------------------------------------------------------------

def split_by_participant(df: pd.DataFrame, verbose: bool = True):
    """
    Split data into train/val/test by participant ID.
    Stratifies by whether the participant has any session reaching Tcore > 39.0.
    """
    # Determine which participants experienced high Tcore
    part_max_tcore = df.groupby('Participant')[TARGET].max()
    high_tcore = (part_max_tcore >= 39.0).astype(int)

    participants = high_tcore.index.values
    labels = high_tcore.values

    # 70% train, 30% temp
    train_parts, temp_parts, _, temp_labels = train_test_split(
        participants, labels,
        test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=labels
    )

    # Split temp into 50/50 val/test (= 15% each overall)
    val_parts, test_parts = train_test_split(
        temp_parts,
        test_size=VAL_FRACTION, random_state=RANDOM_SEED, stratify=temp_labels
    )

    df_train = df[df['Participant'].isin(train_parts)].copy()
    df_val = df[df['Participant'].isin(val_parts)].copy()
    df_test = df[df['Participant'].isin(test_parts)].copy()

    if verbose:
        for name, d, parts in [('Train', df_train, train_parts),
                                ('Val', df_val, val_parts),
                                ('Test', df_test, test_parts)]:
            print(f"  {name}: {len(parts)} participants, {len(d)} rows, "
                  f"{d['Session_ID'].nunique()} sessions")

    return df_train, df_val, df_test


# ---------------------------------------------------------------------------
# 2. IMPUTE — using training-set-only statistics
# ---------------------------------------------------------------------------

def impute_data(df_train, df_val, df_test, verbose=True):
    """
    Impute missing values using training-set statistics only.
    """
    # --- Static: KNN imputation for Body_Fat_Pct ---
    impute_cols = ['Age', 'Sex_Encoded', 'Body_Mass_Kg', 'Height_Cm', 'Body_Fat_Pct']
    impute_cols = [c for c in impute_cols if c in df_train.columns]

    if 'Body_Fat_Pct' in df_train.columns and df_train['Body_Fat_Pct'].isna().any():
        knn = KNNImputer(n_neighbors=KNN_NEIGHBORS)
        df_train[impute_cols] = knn.fit_transform(df_train[impute_cols])
        df_val[impute_cols] = knn.transform(df_val[impute_cols])
        df_test[impute_cols] = knn.transform(df_test[impute_cols])
        # Recompute BMI after imputing Body_Mass_Kg/Height_Cm
        for d in [df_train, df_val, df_test]:
            if 'Body_Mass_Kg' in d.columns and 'Height_Cm' in d.columns:
                height_m = (d['Height_Cm'] / 100).replace(0, np.nan)
                d['BMI'] = (d['Body_Mass_Kg'] / (height_m ** 2)).astype('float32')
        if verbose:
            print("  KNN imputation for Body_Fat_Pct: fit on train, applied to all")

    # --- Dynamic: per-session interpolation + train-median fill ---
    train_medians = df_train[DYNAMIC_RAW_FEATURES].median()

    for name, d in [('train', df_train), ('val', df_val), ('test', df_test)]:
        # Per-session linear interpolation for short gaps
        for sid, session in d.groupby('Session_ID'):
            idx = session.index
            for col in DYNAMIC_RAW_FEATURES:
                d.loc[idx, col] = session[col].interpolate(
                    method='linear', limit=INTERPOLATION_LIMIT, limit_direction='both'
                )

        # Fill remaining NaN with TRAINING medians (not global, not per-participant)
        for col in DYNAMIC_RAW_FEATURES:
            d[col] = d[col].fillna(train_medians[col])

    if verbose:
        total_nan = sum(d[DYNAMIC_RAW_FEATURES].isna().sum().sum()
                        for d in [df_train, df_val, df_test])
        print(f"  Dynamic imputation complete. Remaining NaN: {total_nan}")

    return df_train, df_val, df_test, train_medians


# ---------------------------------------------------------------------------
# 3. ENGINEER — trend features computed per-session AFTER imputation
# ---------------------------------------------------------------------------

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute engineered features per-session. Must be called AFTER imputation.
    """
    result_parts = []
    for sid, session in df.groupby('Session_ID'):
        session = session.sort_values('Time_Sec').copy()
        session['HR_slope_5m'] = session['HR'].diff(periods=5).fillna(0).astype('float32')
        session['SkinTemp_slope_5m'] = session['SkinTemp_UpperArm'].diff(periods=5).fillna(0).astype('float32')
        result_parts.append(session)

    return pd.concat(result_parts).sort_index()


# ---------------------------------------------------------------------------
# 4. SCALE — global StandardScaler fit on training set only
# ---------------------------------------------------------------------------

def scale_data(df_train, df_val, df_test, save_scalers=True, verbose=True):
    """
    StandardScaler fit on training data, applied to all splits.
    Separate scalers for dynamic and static features.
    """
    # Dynamic features (including engineered)
    dyn_scaler = StandardScaler()
    df_train[DYNAMIC_FEATURES] = dyn_scaler.fit_transform(df_train[DYNAMIC_FEATURES])
    df_val[DYNAMIC_FEATURES] = dyn_scaler.transform(df_val[DYNAMIC_FEATURES])
    df_test[DYNAMIC_FEATURES] = dyn_scaler.transform(df_test[DYNAMIC_FEATURES])

    # Static features
    stat_scaler = StandardScaler()
    df_train[STATIC_FEATURES] = stat_scaler.fit_transform(df_train[STATIC_FEATURES])
    df_val[STATIC_FEATURES] = stat_scaler.transform(df_val[STATIC_FEATURES])
    df_test[STATIC_FEATURES] = stat_scaler.transform(df_test[STATIC_FEATURES])

    if save_scalers:
        os.makedirs(os.path.dirname(SCALER_PATH), exist_ok=True)
        with open(SCALER_PATH, 'wb') as f:
            pickle.dump({'dynamic': dyn_scaler, 'static': stat_scaler}, f)
        if verbose:
            print(f"  Scalers saved to {SCALER_PATH}")

    if verbose:
        print(f"  Dynamic scaler: {len(DYNAMIC_FEATURES)} features, "
              f"means={dyn_scaler.mean_.round(2)}")

    return df_train, df_val, df_test, dyn_scaler, stat_scaler


# ---------------------------------------------------------------------------
# 5. WINDOW — create (X_dyn, X_stat, y) arrays
# ---------------------------------------------------------------------------

def create_windows(df: pd.DataFrame, stride: int = 1, window_size: int = None):
    """
    Create sliding windows for the dual-input model.
    """
    if window_size is None:
        window_size = WINDOW_SIZE

    X_dyn, X_stat, y = [], [], []

    for sid, session in df.groupby('Session_ID'):
        session = session.sort_values('Time_Sec')
        if len(session) < window_size:
            continue

        dyn_vals = session[DYNAMIC_FEATURES].values
        stat_val = session[STATIC_FEATURES].iloc[0].values
        targets = session[TARGET].values

        for i in range(0, len(session) - window_size + 1, stride):
            X_dyn.append(dyn_vals[i: i + window_size])
            X_stat.append(stat_val)
            y.append(targets[i + window_size - 1])

    X_dyn = np.array(X_dyn, dtype=np.float32)
    X_stat = np.array(X_stat, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    return X_dyn, X_stat, y


# ---------------------------------------------------------------------------
# Full pipeline: split -> impute -> engineer -> scale -> window
# ---------------------------------------------------------------------------

def prepare_data(df: pd.DataFrame, verbose: bool = True):
    """
    Run the full leak-proof pipeline.
    """
    if verbose:
        print("\n--- Step 1: Splitting by participant ---")
    df_train, df_val, df_test = split_by_participant(df, verbose=verbose)

    participants = {
        'train': sorted(df_train['Participant'].unique()),
        'val': sorted(df_val['Participant'].unique()),
        'test': sorted(df_test['Participant'].unique()),
    }

    if verbose:
        print("\n--- Step 2: Imputing (train stats only) ---")
    df_train, df_val, df_test, train_medians = impute_data(
        df_train, df_val, df_test, verbose=verbose
    )

    if verbose:
        print("\n--- Step 3: Engineering features ---")
    df_train = engineer_features(df_train)
    df_val = engineer_features(df_val)
    df_test = engineer_features(df_test)
    if verbose:
        print(f"  Added {ENGINEERED_FEATURES} per-session")

    if verbose:
        print("\n--- Step 4: Scaling (train-fit only) ---")
    df_train, df_val, df_test, dyn_scaler, stat_scaler = scale_data(
        df_train, df_val, df_test, verbose=verbose
    )

    if verbose:
        print("\n--- Step 5: Windowing ---")

    X_tr_d, X_tr_s, y_tr = create_windows(df_train, stride=TRAIN_STRIDE)
    X_va_d, X_va_s, y_va = create_windows(df_val, stride=EVAL_STRIDE)
    X_te_d, X_te_s, y_te = create_windows(df_test, stride=EVAL_STRIDE)

    if verbose:
        print(f"  Train: {X_tr_d.shape[0]} windows (stride={TRAIN_STRIDE})")
        print(f"  Val:   {X_va_d.shape[0]} windows (stride={EVAL_STRIDE})")
        print(f"  Test:  {X_te_d.shape[0]} windows (stride={EVAL_STRIDE})")
        print(f"  Window shape: ({WINDOW_SIZE}, {X_tr_d.shape[2]}) dynamic + "
              f"({X_tr_s.shape[1]},) static -> 1 Tcore value")

    return {
        'train': (X_tr_d, X_tr_s, y_tr),
        'val': (X_va_d, X_va_s, y_va),
        'test': (X_te_d, X_te_s, y_te),
        'scalers': (dyn_scaler, stat_scaler),
        'train_medians': train_medians,
        'participants': participants,
    }

# --- Execution Block ---
# Runs the pipeline if the data is available in memory
if __name__ == '__main__':
    try:
        # Assumes df was created by running load_data() in the previous cell
        if 'df' not in locals() and 'df' not in globals():
            df = load_data()

        data_splits = prepare_data(df)
        print("\nPipeline complete. Your 'data_splits' dictionary is ready for training!")
    except NameError as e:
        print(f"\n[ERROR] {e}. Ensure you have run the Configuration AND Data Pipeline cells first!")


--- Step 1: Splitting by participant ---
  Train: 15 participants, 6738 rows, 63 sessions
  Val: 3 participants, 1844 rows, 17 sessions
  Test: 4 participants, 2386 rows, 22 sessions

--- Step 2: Imputing (train stats only) ---
  KNN imputation for Body_Fat_Pct: fit on train, applied to all
  Dynamic imputation complete. Remaining NaN: 0

--- Step 3: Engineering features ---
  Added ['HR_slope_5m', 'SkinTemp_slope_5m'] per-session

--- Step 4: Scaling (train-fit only) ---
  Scalers saved to /content/drive/MyDrive/HeatSense/output/models/scalers.pkl
  Dynamic scaler: 6 features, means=[123.81  36.35  34.18  33.75   3.86   0.21]

--- Step 5: Windowing ---
  Train: 4911 windows (stride=1)
  Val:   54 windows (stride=30)
  Test:  71 windows (stride=30)
  Window shape: (30, 6) dynamic + (6,) static -> 1 Tcore value

Pipeline complete. Your 'data_splits' dictionary is ready for training!
